# 03b · Staff usage

Teaching staff use the bot for different reasons from students: demonstrating it in class, testing it,
sending announcements, looking something up while teaching. So they are reported here, on their own
(ANLY-9), and left out of every student measure in notebooks 00–04.

A person is **staff** in a course when they hold an active (not revoked) membership in the course's
organization, as an owner, instructor or assistant, or when their handle matches the configured
override. Everyone else is a student.

**Privacy.** A staff group can be a single person, so this section is aggregates only: counts of
messages and sessions, the mix of purposes, and the split by interface. There is no per-person detail,
no per-course breakdown and no quotes. The usual five-student cell rule cannot apply to a group this
small, so it is replaced by that restriction (see `docs/DECISIONS.md` D-147).

In [ ]:
# Put the analysis package on the path no matter where Jupyter was started.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "analysis" / "bloombot_analysis").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "analysis"))

import pandas as pd

from bloombot_analysis import charts, load, metrics, privacy, report, sessions, topics
from bloombot_analysis.config import CONFIG, SURFACE_LABELS, STAFF_TOPICS, STUDENT_TOPICS

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print("as of:", CONFIG.as_of)
print("input     :", CONFIG.describe_inputs())
print("output    :", CONFIG.out_dir)

In [ ]:
messages_df = pd.read_csv(CONFIG.data_path("messages.csv"), parse_dates=["ts", "week"])
sessions_df = pd.read_csv(CONFIG.data_path("sessions.csv"), parse_dates=["started_at", "ended_at", "week"])
topics_df = pd.read_csv(CONFIG.data_path("session_topics.csv"))

staff_messages = messages_df[messages_df["role"] == "staff"]
staff_sessions = sessions_df[sessions_df["role"] == "staff"]
staff_topics_df = topics_df[topics_df["role"] == "staff"]
print(len(staff_messages), "staff messages,", len(staff_sessions), "staff sessions")

## How much of the traffic

Shares of everything recorded, student and staff together. *Prompts* are the messages people sent to
the bot (the bot's replies are not counted).

In [ ]:
all_prompts = int((messages_df["direction"] == "from").sum())
staff_prompts = int((staff_messages["direction"] == "from").sum())
share = pd.DataFrame(
    {
        "Staff": [len(staff_messages), len(staff_sessions), staff_prompts],
        "All": [len(messages_df), len(sessions_df), all_prompts],
    },
    index=["Messages", "Sessions", "Prompts"],
)
share["Staff share"] = share["Staff"] / share["All"].where(share["All"] > 0)
share

## What staff used it for

One purpose per session, from the staff label set. The keyword classifier reads only the staff
member's own words; the same caveat applies as for students, so the agreement rate from the hand
audit is reported when one exists.

In [ ]:
purpose = topics.topic_counts(staff_topics_df, STAFF_TOPICS)
purpose_fig = None
if purpose["sessions"].sum():
    purpose_fig = charts.bar_h(
        purpose.sort_values("sessions")["topic"], purpose.sort_values("sessions")["sessions"],
        "staff_purposes", "Staff sessions by purpose", "Sessions",
    )
purpose.sort_values("sessions", ascending=False)

## Where staff used it

In [ ]:
staff_surface = sessions.surface_split(staff_sessions).drop(columns=["students"])
staff_surface["surface_label"] = staff_surface["surface"].map(SURFACE_LABELS).fillna(staff_surface["surface"])
surface_fig = None
if len(staff_surface):
    surface_fig = charts.bar_h(
        staff_surface["surface_label"], staff_surface["sessions"],
        "staff_sessions_by_surface", "Staff sessions by interface", "Sessions",
    )
staff_surface

In [ ]:
completed_path = CONFIG.data_path("topic_audit_completed.csv")
staff_agreement = {"checked": 0, "agreed": 0, "rate": float("nan")}
if completed_path.exists():
    audited = pd.read_csv(completed_path)
    if "role" in audited:
        staff_agreement = topics.agreement_rate(audited[audited["role"] == "staff"])

other = int((staff_topics_df["topic"] == "Other").sum()) if len(staff_topics_df) else 0
metrics.update("staff", {
    "messages": len(staff_messages),
    "sessions": len(staff_sessions),
    "prompts": staff_prompts,
    "staff_people": int(staff_sessions["person_key"].nunique()) if len(staff_sessions) else 0,
    "all_messages": len(messages_df),
    "all_sessions": len(sessions_df),
    "all_prompts": all_prompts,
    "message_share": float(len(staff_messages) / len(messages_df)) if len(messages_df) else None,
    "session_share": float(len(staff_sessions) / len(sessions_df)) if len(sessions_df) else None,
    "prompt_share": float(staff_prompts / all_prompts) if all_prompts else None,
    "purposes": purpose.sort_values("sessions", ascending=False).to_dict(orient="records"),
    "other_share": other / len(staff_topics_df) if len(staff_topics_df) else None,
    "by_surface": staff_surface.to_dict(orient="records"),
    "agreement": staff_agreement,
    "figures": {
        "purposes": purpose_fig.name if purpose_fig is not None else None,
        "surface": surface_fig.name if surface_fig is not None else None,
    },
})
print("ok")